### Batch 1 – Initial load

| Record type | Count |
| :--- | :---: |
| Valid unique orders | 130 |
| Duplicate rows | 6 |
| Invalid rows | 4 |
| **Total** | **140** |


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    IntegerType, DecimalType
)

from pyspark.sql.functions import (
    lit, current_timestamp, col
)

# Source file path
source_path = "/Volumes/final_assesment/source/online_shopping_files/Batch_01_Online_Shopping.csv"

# Target Bronze table
target_table = "final_assesment.bronze.orders_raw"

# Batch identification
batch_id = "Batch_01"

source_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_updated_at", TimestampType(), True),

    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
    StructField("customer_dob", DateType(), True),

    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),

    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DecimalType(12, 2), True),
    StructField("discount", DecimalType(12, 2), True),

    StructField("payment_method_id", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("payment_provider", StringType(), True)
])

print("Source schema defined successfully.")

In [0]:
df_source = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("dateFormat", "yyyy-MM-dd")
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .schema(source_schema)
    .load(source_path)
)

print("CSV file read successfully.")
print("Source record count:", df_source.count())

display(df_source)

In [0]:
df_bronze = (
    df_source
    .withColumn("batch_id", lit(batch_id))
    .withColumn("source_file", lit("Batch_01_Online_Shopping.csv"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

display(df_bronze)

In [0]:
# If Batch 1 has already been successfully loaded, the notebook will skip it instead of appending the same batch again.

if spark.catalog.tableExists(target_table):

    existing_batch = (
        spark.table(target_table)
        .filter(col("batch_id") == batch_id)
        .limit(1)
        .count()
    )

    if existing_batch > 0:
        print(f"{batch_id} is already ingested. Skipping ingestion.")

    else:
        (
            df_bronze.write
            .format("delta")
            .mode("append")
            .saveAsTable(target_table)
        )

        print(f"{batch_id} ingested successfully.")

else:

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    print(f"Bronze table created and {batch_id} ingested successfully.")

In [0]:
%sql
SELECT
    batch_id,
    source_file,
    COUNT(*) AS total_records,
    COUNT(DISTINCT order_id) AS distinct_order_ids,
    MIN(ingestion_timestamp) AS first_ingestion,
    MAX(ingestion_timestamp) AS last_ingestion
FROM final_assesment.bronze.orders_raw
GROUP BY
    batch_id,
    source_file;

In [0]:
%sql
SELECT *
FROM final_assesment.bronze.orders_raw
LIMIT 20;

In [0]:
%sql
SELECT *
FROM final_assesment.bronze.orders_raw
LIMIT 20;